# Volume 9, Chapter 9: The Symmetric Group (Python companion)

The Robinson–Schensted correspondence and the hook length formula, Specht modules and the character table of S5 by computer, the hook walk, and longest increasing subsequences from Ulam's problem to patience diff.

*Before each solution, predict what the output will look like.*

## Exercises

**P9.1 (★) The Robinson–Schensted correspondence.** Implement row insertion, the map $\sigma\mapsto(P(\sigma),Q(\sigma))$ of Theorem 9.11, and its inverse by reverse bumping. (a) Check on all of $\mathfrak S_n$, $n\le7$, that the map is a bijection onto pairs of standard tableaux of equal shape, and that $\sum_\lambda(f^\lambda)^2=n!$. (b) Check the hook length formula (Theorem 9.14) against the recursion (9.1) for all partitions of $n\le14$.

**P9.2 (★★) Specht modules by computer.** For each $\lambda\vdash5$, build the matrix whose columns are all the polytabloids $e_T$ in the basis of tabloids. (a) Check that its rank is $f^\lambda$ and that the standard polytabloids are already independent (Theorem 9.9). (b) Compute the character of $S^\lambda$ on every conjugacy class as the trace of $\sigma$ composed with the orthogonal projection onto $S^\lambda$, check the orthogonality relations, and compare the transposition column with the content formula (Theorem 9.15).

**P9.3 (★) The hook walk.** Simulate the Greene–Nijenhuis–Wilf hook walk on $\lambda=(5,4,2,1)$ (Figure 9.1) and compare the frequency of each corner with $F(\lambda-c)/F(\lambda)=f^{\lambda-c}/f^\lambda$.

**P9.4 (★★) CS application: longest increasing subsequences and patience diff.** (a) Compute the longest increasing subsequence $L_n$ of random permutations in $O(n\log n)$ time by patience sorting (Exercise 9.5) for $n$ up to $10^6$, and compare $L_n$ with $2\sqrt n$ and $(L_n-2\sqrt n)/n^{1/6}$ with the mean $-1.77$ of the Tracy–Widom law (beauty box). (b) The *patience diff* algorithm of Bram Cohen (used by Bazaar, and by `git diff --patience`) aligns two versions of a file by taking the lines that occur exactly once in each, and keeping the longest increasing subsequence of their positions as anchors. Implement it and run it on two versions of a short program.

## Solutions

### P9.1

In [ ]:
import itertools, math, bisect
from collections import Counter

def insert(P, x):
    """Row-insert x into P (a list of rows); return the row index of the new cell."""
    r = 0
    while True:
        if r == len(P):
            P.append([x]); return r
        row = P[r]
        k = bisect.bisect_right(row, x)
        if k == len(row):
            row.append(x); return r
        x, row[k] = row[k], x
        r += 1

def rs(sigma):
    P, Q = [], []
    for k, x in enumerate(sigma, 1):
        r = insert(P, x)
        if r == len(Q): Q.append([])
        Q[r].append(k)
    return P, Q

def rs_inverse(P, Q):
    P = [row[:] for row in P]; Q = [row[:] for row in Q]
    n = sum(map(len, P)); sigma = [0] * n
    for k in range(n, 0, -1):
        r = next(i for i, row in enumerate(Q) if row and row[-1] == k)   # the cell added at step k
        Q[r].pop(); y = P[r].pop()
        for i in range(r - 1, -1, -1):                                   # reverse bumping
            row = P[i]
            j = bisect.bisect_left(row, y) - 1                           # largest entry smaller than y
            y, row[j] = row[j], y
        sigma[k - 1] = y
        while P and not P[-1]: P.pop(); Q.pop()
    return tuple(sigma)

print(rs((3, 1, 4, 5, 2)))
for n in range(1, 8):
    shapes, ok = Counter(), True
    for sigma in itertools.permutations(range(1, n + 1)):
        P, Q = rs(sigma)
        shapes[tuple(map(len, P))] += 1
        ok &= rs_inverse(P, Q) == sigma
    # the number of permutations with shape lambda should be (f^lambda)^2
    squares = all(math.isqrt(c) ** 2 == c for c in shapes.values())
    print(n, "inverse ok:", ok, " partitions:", len(shapes), " counts are squares:", squares,
          " sum:", sum(shapes.values()), "=", math.factorial(n))

The example $31452$ of the text gives $P=125/34$ and $Q=134/25$. For each $n\le7$ reverse bumping recovers every permutation, every partition of $n$ occurs as a shape ($1,2,3,5,7,11,15$ of them), and the number of permutations with a given shape is a perfect square, namely $(f^\lambda)^2$: the bijection is onto pairs of tableaux of equal shape. Now the hook length formula against the recursion "remove the corner holding $n$".

In [ ]:
from functools import lru_cache

def partitions(n, maxpart=None):
    maxpart = n if maxpart is None else maxpart
    if n == 0: yield (); return
    for k in range(min(n, maxpart), 0, -1):
        for rest in partitions(n - k, k): yield (k,) + rest

def conjugate(lam): return tuple(sum(1 for l in lam if l > j) for j in range(lam[0])) if lam else ()

def hook_formula(lam):
    c = conjugate(lam); prod = 1
    for i, l in enumerate(lam):
        for j in range(l): prod *= (l - j - 1) + (c[j] - i - 1) + 1
    return math.factorial(sum(lam)) // prod

@lru_cache(None)
def f_recursive(lam):
    if sum(lam) <= 1: return 1
    total = 0
    for i in range(len(lam)):
        if i == len(lam) - 1 or lam[i] > lam[i + 1]:                     # (i, lam[i]) is a corner
            mu = list(lam); mu[i] -= 1
            total += f_recursive(tuple(x for x in mu if x))
    return total

bad = [lam for n in range(1, 15) for lam in partitions(n) if hook_formula(lam) != f_recursive(lam)]
print("partitions checked:", sum(1 for n in range(1, 15) for _ in partitions(n)), " mismatches:", bad)
print("f(5,4,2,1) =", hook_formula((5, 4, 2, 1)), "  f(3,2,1) =", hook_formula((3, 2, 1)))
print("sum of f^2 for n = 14:", sum(hook_formula(l) ** 2 for l in partitions(14)) == math.factorial(14))

All $507$ partitions of $n\le14$ agree, the two examples of the text come out as $5775$ and $16$, and $\sum(f^\lambda)^2=14!$, which is far beyond what we could check by running through $\mathfrak S_{14}$ (87 billion permutations).

### P9.2

In [ ]:
import numpy as np

def tableaux(lam):
    """All tableaux of shape lam, as tuples of rows."""
    n = sum(lam)
    for perm in itertools.permutations(range(1, n + 1)):
        rows, k = [], 0
        for l in lam: rows.append(perm[k:k + l]); k += l
        yield tuple(rows)

def tabloid(T): return tuple(frozenset(r) for r in T)
def columns(T): return [[row[j] for row in T if j < len(row)] for j in range(len(T[0]))]
def is_standard(T):
    return all(list(r) == sorted(r) for r in T) and all(c == sorted(c) for c in columns(T))
def perm_sign(p):                     # p: dict a -> p(a)
    seen, s = set(), 1
    for a in p:
        if a in seen: continue
        b, length = a, 0
        while b not in seen: seen.add(b); b = p[b]; length += 1
        s *= (-1) ** (length - 1)
    return s

def column_group(T):
    cols = columns(T)
    for images in itertools.product(*[itertools.permutations(c) for c in cols]):
        p = {}
        for c, im in zip(cols, images): p.update(zip(c, im))
        yield p

def specht_data(lam):
    n = sum(lam)
    tabs = list(tableaux(lam))
    index = {}
    for T in tabs: index.setdefault(tabloid(T), len(index))
    def polytabloid(T):
        v = np.zeros(len(index))
        for p in column_group(T):
            v[index[tabloid(tuple(tuple(p[a] for a in r) for r in T))]] += perm_sign(p)
        return v
    reps = {tabloid(T): T for T in tabs}                       # one tableau per tabloid, to get columns
    all_e = np.array([polytabloid(T) for T in tabs]).T
    std_e = np.array([polytabloid(T) for T in tabs if is_standard(T)]).T
    return index, all_e, std_e

def act(sigma, index):
    """Matrix of a permutation (tuple, sigma[a-1] = image of a) on the tabloid basis."""
    M = np.zeros((len(index), len(index)))
    for t, i in index.items():
        M[index[tuple(frozenset(sigma[a - 1] for a in r) for r in t)], i] = 1
    return M

n = 5
classes = {"1": (1, 2, 3, 4, 5), "(12)": (2, 1, 3, 4, 5), "(12)(34)": (2, 1, 4, 3, 5), "(123)": (2, 3, 1, 4, 5),
           "(123)(45)": (2, 3, 1, 5, 4), "(1234)": (2, 3, 4, 1, 5), "(12345)": (2, 3, 4, 5, 1)}
sizes = {"1": 1, "(12)": 10, "(12)(34)": 15, "(123)": 20, "(123)(45)": 20, "(1234)": 30, "(12345)": 24}
table = {}
for lam in partitions(n):
    index, all_e, std_e = specht_data(lam)
    U, s, _ = np.linalg.svd(all_e)
    rank = int((s > 1e-9).sum()); B = U[:, :rank]               # orthonormal basis of S^lambda
    Proj = B @ B.T
    chi = [round(np.trace(act(g, index) @ Proj)) for g in classes.values()]
    table[lam] = chi
    contents = sum(j - i for i, l in enumerate(lam) for j in range(l))
    print(f"{str(lam):16} dim M = {len(index):3}  rank = {rank:2}  f = {hook_formula(lam):2}  "
          f"standard independent: {np.linalg.matrix_rank(std_e) == std_e.shape[1]}  chi = {chi}  "
          f"content formula: {hook_formula(lam) * contents / math.comb(n, 2):5.1f}")

For each of the seven partitions of $5$, the polytabloids span a space of dimension exactly $f^\lambda$ ($1,4,5,6,5,4,1$), and the $f^\lambda$ standard ones are independent, so they are a basis, as Theorems 9.9 and 9.11 say. The traces give the whole character table of $\mathfrak S_5$, and the column of $(12)$ is $f^\lambda$ times the content sum divided by $\binom52=10$. Note that $(3,1,1)$, which is its own conjugate, has $\chi(\tau)=0$ (Corollary 9.16). Last, the orthogonality relations:

In [ ]:
X = np.array(list(table.values()), dtype=float)
w = np.array(list(sizes.values())) / 120
print("rows orthonormal:", np.allclose((X * w) @ X.T, np.eye(len(X))))

So the Specht modules are irreducible and pairwise distinct, exactly as Theorem 9.7 predicts.

### P9.3

In [ ]:
import random

def hook_walk(lam, rng):
    c = conjugate(lam)
    cells = [(i, j) for i in range(len(lam)) for j in range(lam[i])]
    i, j = rng.choice(cells)
    while True:
        arm, leg = lam[i] - j - 1, c[j] - i - 1
        if arm + leg == 0: return (i + 1, j + 1)             # a corner, in 1-based coordinates
        k = rng.randrange(arm + leg)
        if k < arm: j += k + 1
        else: i += k - arm + 1

lam = (5, 4, 2, 1); rng = random.Random(9); N = 200_000
counts = Counter(hook_walk(lam, rng) for _ in range(N))
f = hook_formula(lam)
for (a, b) in sorted(counts):
    mu = list(lam); mu[a - 1] -= 1; mu = tuple(x for x in mu if x)
    print(f"corner {(a, b)}: simulated {counts[(a, b)] / N:.4f}   f(lambda - c)/f(lambda) = {hook_formula(mu)}/{f} = {hook_formula(mu) / f:.4f}")

The four frequencies agree with $f^{\lambda-c}/f^\lambda$ to within the sampling error (about $0.001$ for $200\,000$ runs), and the four exact values add up to $1$: this is (9.2), the identity at the heart of the proof.

### P9.4

In [ ]:
import numpy as np, time

def lis_length(seq):
    tops = []                                     # the pile tops = first row of P
    for x in seq:
        k = bisect.bisect_left(tops, x)
        if k == len(tops): tops.append(x)
        else: tops[k] = x
    return len(tops)

rng = np.random.default_rng(2026)
for n in [10**2, 10**3, 10**4, 10**5, 10**6]:
    trials = 200 if n <= 10**4 else (40 if n == 10**5 else 8)
    t = time.time()
    L = np.array([lis_length(rng.permutation(n).tolist()) for _ in range(trials)])
    scaled = (L - 2 * np.sqrt(n)) / n ** (1 / 6)
    print(f"n = {n:>8}  mean L = {L.mean():9.1f}  2 sqrt(n) = {2*np.sqrt(n):7.0f}  L/sqrt(n) = {L.mean()/np.sqrt(n):.3f}"
          f"  mean (L - 2 sqrt n)/n^(1/6) = {scaled.mean():6.2f}   ({time.time() - t:.1f} s)")

$L_n/\sqrt n$ creeps up towards $2$ (Logan–Shepp, Vershik–Kerov), and the gap $2\sqrt n-L_n$ grows only like $n^{1/6}$: the rescaled gap stays between about $-1.5$ and $-1.8$, drifting towards the Tracy–Widom mean $-1.77$ as $n$ grows (Baik–Deift–Johansson); the convergence is slow, and with only $8$ trials the value at $n=10^6$ still fluctuates by about $\pm0.3$. Patience sorting handles a million cards in a few seconds, since each card costs one binary search.

Now patience diff. The classical diff computes a longest common subsequence of lines, which aligns blank lines and lone braces wherever it can, and sometimes produces confusing output. Bram Cohen's idea is to anchor only on lines that are *unique* in both files, which are most likely to be "the same line"; among those, the order-preserving ones are a longest increasing subsequence of their positions in the second file, listed in the order of the first. Between consecutive anchors one recurses, and a full implementation falls back on classical diff inside the gaps; we keep it short and simply report the gaps.

In [ ]:
def lis_indices(seq):
    """Indices of one longest increasing subsequence of seq, by patience sorting with back-pointers."""
    tops, top_idx, back = [], [], [None] * len(seq)
    for i, x in enumerate(seq):
        k = bisect.bisect_left(tops, x)
        back[i] = top_idx[k - 1] if k else None
        if k == len(tops): tops.append(x); top_idx.append(i)
        else: tops[k] = x; top_idx[k] = i
    out, i = [], top_idx[-1] if top_idx else None
    while i is not None: out.append(i); i = back[i]
    return out[::-1]

def patience_diff(a, b, a0=0, b0=0, out=None):
    out = [] if out is None else out
    ca, cb = Counter(a), Counter(b)
    uniq = [(i, b.index(line)) for i, line in enumerate(a) if ca[line] == 1 and cb[line] == 1]
    if not uniq:                                    # no anchor: report the whole gap
        out += [f"- {l}" for l in a] + [f"+ {l}" for l in b]
        return out
    anchors = [uniq[k] for k in lis_indices([j for _, j in uniq])]
    pi, pj = 0, 0
    for i, j in anchors + [(len(a), len(b))]:
        if (pi, pj) != (i, j) and (a[pi:i] or b[pj:j]):
            patience_diff(a[pi:i], b[pj:j], a0 + pi, b0 + pj, out)
        if i < len(a): out.append(f"  {a[i]}")
        pi, pj = i + 1, j + 1
    return out

old = """def mean(xs):
    total = 0
    for x in xs:
        total += x
    return total / len(xs)

def var(xs):
    m = mean(xs)
    return mean([(x - m) ** 2 for x in xs])
""".splitlines()
new = """def var(xs):
    m = mean(xs)
    return mean([(x - m) ** 2 for x in xs])

def mean(xs):
    if not xs:
        raise ValueError("empty")
    return sum(xs) / len(xs)
""".splitlines()
print("\n".join(patience_diff(old, new)))

The function `var` was moved above `mean`, and `mean` was rewritten. The unique lines common to both files are the three lines of `var` and the line `def mean(xs):`; their positions in the new file, read in the order of the old file, are $4,0,1,2$ (counting from $0$), and the longest increasing subsequence keeps $0,1,2$: the three lines of `var` are the anchors, and `def mean(xs):` is reported as deleted and re-added. So the diff shows `mean` as removed from the top and a new `mean` at the bottom, with `var` untouched, which is how a human would describe the change. A plain longest-common-subsequence diff is free to match the blank line and `return` fragments instead, and can interleave the old and new bodies. Here Schensted's theorem, the first row of the RSK tableau, is a tool that programmers use every day.